# Study the laplacian regression on the Larged Uncrompressed Graph

In [3]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import networkx as nx

from scipy import sparse
from scipy.sparse.linalg import spsolve

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


# ============================================================
# 1. FILE LOCATIONS
# ============================================================

BASE_DIR = Path.cwd()

FULL_GRAPH_FILE = (
    BASE_DIR
    / "largest_river_component.pkl"
)

MASTER_GRAPH_FILE = (
    BASE_DIR
    / "master_compressed_river_graph.pkl"
)

HOURLY_FILE = (
    BASE_DIR
    / "usgs_all_stations_hourly.csv"
)


print("Current directory:")
print(BASE_DIR)


# ============================================================
# 2. CHECK FILES EXIST
# ============================================================

required_files = [
    FULL_GRAPH_FILE,
    MASTER_GRAPH_FILE,
    HOURLY_FILE
]


for file in required_files:

    print(
        file.name,
        "->",
        "FOUND"
        if file.exists()
        else "MISSING"
    )


missing_files = [
    file
    for file in required_files
    if not file.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Missing files:\n"
        +
        "\n".join(
            str(file)
            for file in missing_files
        )
    )


# ============================================================
# 3. LOAD UNCOMPRESSED LARGEST COMPONENT
# ============================================================

with open(
    FULL_GRAPH_FILE,
    "rb"
) as f:

    G_full = pickle.load(f)


print("\n" + "=" * 60)
print("FULL / UNCOMPRESSED GRAPH")
print("=" * 60)

print(
    "Directed:",
    G_full.is_directed()
)

print(
    "Nodes:",
    G_full.number_of_nodes()
)

print(
    "Edges:",
    G_full.number_of_edges()
)


if G_full.is_directed():

    print(
        "Weak components:",
        nx.number_weakly_connected_components(
            G_full
        )
    )

else:

    print(
        "Connected components:",
        nx.number_connected_components(
            G_full
        )
    )


# ============================================================
# 4. LOAD MASTER COMPRESSED GRAPH
#
# We are NOT using this for the new Laplacian.
# We only need its station/support metadata.
# ============================================================

with open(
    MASTER_GRAPH_FILE,
    "rb"
) as f:

    G_master = pickle.load(f)


print("\n" + "=" * 60)
print("MASTER METADATA GRAPH")
print("=" * 60)

print(
    "Nodes:",
    G_master.number_of_nodes()
)

print(
    "Edges:",
    G_master.number_of_edges()
)


# ============================================================
# 5. LOAD HOURLY USGS DATA
# ============================================================

hourly = pd.read_csv(
    HOURLY_FILE
)


hourly["time"] = pd.to_datetime(
    hourly["time"],
    utc=True,
    errors="coerce"
)


print("\n" + "=" * 60)
print("HOURLY DATA")
print("=" * 60)

print(
    "Rows:",
    len(hourly)
)

print(
    "Stations:",
    hourly["station_id"].nunique()
)

print(
    "Columns:"
)

print(
    hourly.columns.tolist()
)


# ============================================================
# 6. BUILD STATION -> GRAPH NODE MAPPING
# ============================================================

station_to_node = {}


for node, data in G_master.nodes(
    data=True
):

    clusters = data.get(
        "usgs_clusters",
        []
    )


    for cluster in clusters:

        stations = cluster.get(
            "member_stations",
            []
        )


        # fallback
        if not stations:

            anchor = cluster.get(
                "anchor_station"
            )

            if anchor is not None:

                stations = [
                    anchor
                ]


        for station in stations:

            station_to_node[
                station
            ] = node


print(
    "\nWQ stations mapped:",
    len(station_to_node)
)


# ============================================================
# 7. IDENTIFY WQ NODES
# ============================================================

all_wq_nodes = {

    node

    for node, data
    in G_master.nodes(
        data=True
    )

    if (
        data.get(
            "usgs_station_count",
            0
        ) > 0

        or

        len(
            data.get(
                "usgs_clusters",
                []
            )
        ) > 0
    )
}


# ============================================================
# 8. IDENTIFY SUPPORT-GAUGE NODES
# ============================================================

support_nodes = {

    node

    for node, data
    in G_master.nodes(
        data=True
    )

    if (
        data.get(
            "support_gauge_count",
            0
        ) > 0

        or

        len(
            data.get(
                "support_gauges",
                []
            )
        ) > 0
    )
}


print(
    "WQ graph nodes:",
    len(all_wq_nodes)
)

print(
    "Support-gauge nodes:",
    len(support_nodes)
)


# ============================================================
# 9. VERIFY THEY EXIST IN FULL GRAPH
# ============================================================

full_nodes = set(
    G_full.nodes()
)


missing_wq = (
    all_wq_nodes
    - full_nodes
)

missing_support = (
    support_nodes
    - full_nodes
)


print("\n" + "=" * 60)
print("FULL-GRAPH MONITORING CHECK")
print("=" * 60)


print(
    "WQ nodes present:",
    len(
        all_wq_nodes
        &
        full_nodes
    ),
    "/",
    len(all_wq_nodes)
)


print(
    "Support nodes present:",
    len(
        support_nodes
        &
        full_nodes
    ),
    "/",
    len(support_nodes)
)


print(
    "\nMissing WQ nodes:",
    sorted(
        missing_wq
    )
)

print(
    "Missing support nodes:",
    sorted(
        missing_support
    )
)


# ============================================================
# 10. ATTACH GRAPH NODE TO HOURLY DATA
# ============================================================

hourly["node"] = (
    hourly["station_id"]
    .map(
        station_to_node
    )
)


mapped_hourly = (
    hourly[
        hourly["node"]
        .notna()
    ]
    .copy()
)


mapped_hourly[
    "node"
] = (
    mapped_hourly[
        "node"
    ]
    .astype(int)
)


print("\n" + "=" * 60)
print("READY FOR MODELING")
print("=" * 60)

print(
    "Mapped hourly rows:",
    len(mapped_hourly)
)

print(
    "Mapped WQ graph nodes:",
    mapped_hourly[
        "node"
    ].nunique()
)

print(
    "Full graph nodes available to Laplacian:",
    G_full.number_of_nodes()
)

Current directory:
/Users/andrewchincea/TDA_Lab_Practice/rivernets/making_the_graph
largest_river_component.pkl -> FOUND
master_compressed_river_graph.pkl -> FOUND
usgs_all_stations_hourly.csv -> FOUND

FULL / UNCOMPRESSED GRAPH
Directed: True
Nodes: 67401
Edges: 67400
Weak components: 1

MASTER METADATA GRAPH
Nodes: 67
Edges: 66

HOURLY DATA
Rows: 234095
Stations: 28
Columns:
['station_id', 'time', 'DO concentration', 'Water temperature', 'Specific conductance', 'Turbidity', 'pH', 'Discharge', 'Stage']

WQ stations mapped: 28
WQ graph nodes: 26
Support-gauge nodes: 3

FULL-GRAPH MONITORING CHECK
WQ nodes present: 26 / 26
Support nodes present: 3 / 3

Missing WQ nodes: []
Missing support nodes: []

READY FOR MODELING
Mapped hourly rows: 234095
Mapped WQ graph nodes: 26
Full graph nodes available to Laplacian: 67401


In [4]:
# ============================================================
# Graph:
#     67,401-node uncompressed largest component
#
# Edge weight:
#     w_ij = 1 / distance_km
#
# Standard undirected combinatorial Laplacian:
#     L = D - W
# ============================================================

from scipy import sparse
from math import radians, sin, cos, asin, sqrt


# ============================================================
# 1. HAVERSINE DISTANCE
# ============================================================

def haversine_m(
    lon1,
    lat1,
    lon2,
    lat2
):

    R = 6371000.0

    lon1 = radians(float(lon1))
    lat1 = radians(float(lat1))
    lon2 = radians(float(lon2))
    lat2 = radians(float(lat2))

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        sin(dlat / 2.0) ** 2
        +
        cos(lat1)
        * cos(lat2)
        * sin(dlon / 2.0) ** 2
    )

    return (
        2.0
        * R
        * asin(
            sqrt(a)
        )
    )


# ============================================================
# 2. FIXED NODE ORDER
# ============================================================

full_node_order = list(
    G_full.nodes()
)

full_node_index = {
    node: i
    for i, node
    in enumerate(full_node_order)
}

N_FULL = len(
    full_node_order
)

print(
    "Full graph node count:",
    N_FULL
)


# ============================================================
# 3. BUILD SPARSE ADJACENCY
# ============================================================

rows = []
cols = []
weights = []

edge_distances_km = []


# Treat directed river graph as undirected for standard
# symmetric Graph-Laplacian regression.
G_full_und = G_full.to_undirected(
    as_view=True
)


for u, v, data in G_full_und.edges(
    data=True
):

    # --------------------------------------------------------
    # Physical edge distance
    # --------------------------------------------------------

    length_m = data.get(
        "length_m",
        None
    )


    if (
        length_m is None
        or not np.isfinite(
            float(length_m)
        )
        or float(length_m) <= 0
    ):

        udata = G_full.nodes[u]
        vdata = G_full.nodes[v]

        length_m = haversine_m(

            udata["longitude"],
            udata["latitude"],

            vdata["longitude"],
            vdata["latitude"]
        )


    distance_km = max(
        float(length_m) / 1000.0,
        1e-8
    )


    edge_distances_km.append(
        distance_km
    )


    # --------------------------------------------------------
    # Conductance-style edge weight
    # --------------------------------------------------------

    w = (
        1.0
        /
        distance_km
    )


    i = full_node_index[u]
    j = full_node_index[v]


    rows.extend(
        [
            i,
            j
        ]
    )

    cols.extend(
        [
            j,
            i
        ]
    )

    weights.extend(
        [
            w,
            w
        ]
    )


# ============================================================
# 4. SPARSE W
# ============================================================

W_full = sparse.coo_matrix(

    (
        weights,
        (
            rows,
            cols
        )
    ),

    shape=(
        N_FULL,
        N_FULL
    )

).tocsr()


# ============================================================
# 5. SPARSE LAPLACIAN
# ============================================================

degree = np.asarray(
    W_full.sum(
        axis=1
    )
).ravel()


L_full = (
    sparse.diags(
        degree
    )
    -
    W_full
).tocsr()


# ============================================================
# 6. CHECKS
# ============================================================

edge_distances_km = np.asarray(
    edge_distances_km
)


print(
    "\nLaplacian shape:",
    L_full.shape
)

print(
    "Nonzero entries:",
    L_full.nnz
)


storage_mb = (

    L_full.data.nbytes
    +
    L_full.indices.nbytes
    +
    L_full.indptr.nbytes

) / 1024**2


print(
    "Approx sparse storage:",
    round(storage_mb, 2),
    "MB"
)


print(
    "\nEdge distance km:"
)

print(
    pd.Series(
        edge_distances_km
    ).describe()
)


symmetry_error = (
    L_full
    -
    L_full.T
)


print(
    "\nSymmetric:",
    symmetry_error.nnz == 0
)


print(
    "Finite:",
    np.isfinite(
        L_full.data
    ).all()
)


# Row sums should be approximately zero
row_sums = np.asarray(
    L_full.sum(
        axis=1
    )
).ravel()


print(
    "Maximum absolute row sum:",
    np.abs(
        row_sums
    ).max()
)

Full graph node count: 67401

Laplacian shape: (67401, 67401)
Nonzero entries: 202201
Approx sparse storage: 2.57 MB

Edge distance km:
count    67400.000000
mean         0.097055
std          0.019850
min          0.068725
25%          0.076338
50%          0.092662
75%          0.118483
max          0.121598
dtype: float64

Symmetric: True
Finite: True
Maximum absolute row sum: 7.105427357601002e-15


In [5]:
from itertools import combinations


WQ_VARIABLES = [
    "DO concentration",
    "Water temperature",
    "Specific conductance",
    "Turbidity",
    "pH"
]


# ============================================================
# 1. COLLAPSE TO ONE ROW PER GRAPH NODE / HOUR
# ============================================================

node_hourly_all = (
    mapped_hourly[
        [
            "node",
            "time"
        ]
        + WQ_VARIABLES
    ]
    .groupby(
        [
            "node",
            "time"
        ],
        as_index=False
    )[
        WQ_VARIABLES
    ]
    .median()
)


print(
    "Node-hour rows:",
    len(node_hourly_all)
)

print(
    "WQ graph nodes represented:",
    node_hourly_all[
        "node"
    ].nunique()
)


# ============================================================
# 2. TEST EVERY PREDICTOR SUBSET FOR EVERY TARGET
# ============================================================

coverage_rows = []


for target in WQ_VARIABLES:

    candidate_features = [
        variable
        for variable in WQ_VARIABLES
        if variable != target
    ]


    # Every node that has the target at least once
    target_nodes = set(
        node_hourly_all.loc[
            node_hourly_all[
                target
            ].notna(),
            "node"
        ].unique()
    )


    for r in range(
        1,
        len(candidate_features) + 1
    ):

        for combo in combinations(
            candidate_features,
            r
        ):

            features = list(
                combo
            )


            usable = (
                node_hourly_all
                .dropna(
                    subset=[
                        target
                    ] + features
                )
            )


            counts = (
                usable
                .groupby(
                    "node"
                )
                .size()
            )


            usable_nodes = set(
                counts.index
            )


            missing_nodes = sorted(
                target_nodes
                -
                usable_nodes
            )


            coverage_rows.append({

                "target":
                    target,

                "features":
                    " + ".join(
                        features
                    ),

                "n_features":
                    len(
                        features
                    ),

                "target_nodes":
                    len(
                        target_nodes
                    ),

                "usable_nodes":
                    len(
                        usable_nodes
                    ),

                "all_target_nodes_covered":
                    (
                        usable_nodes
                        ==
                        target_nodes
                    ),

                "total_node_hours":
                    len(
                        usable
                    ),

                "minimum_hours_per_node":
                    (
                        int(
                            counts.min()
                        )
                        if len(counts)
                        else 0
                    ),

                "missing_nodes":
                    missing_nodes
            })


coverage_table = pd.DataFrame(
    coverage_rows
)


# ============================================================
# 3. DISPLAY BEST OPTIONS FOR EACH TARGET
# ============================================================

for target in WQ_VARIABLES:

    print(
        "\n"
        + "=" * 75
    )

    print(
        "TARGET:",
        target
    )

    print(
        "=" * 75
    )


    target_table = (
        coverage_table[
            coverage_table[
                "target"
            ] == target
        ]
        .sort_values(
            [
                "all_target_nodes_covered",
                "usable_nodes",
                "n_features",
                "total_node_hours"
            ],
            ascending=[
                False,
                False,
                False,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )


    display(
        target_table.head(
            12
        )
    )

Node-hour rows: 225339
WQ graph nodes represented: 26

TARGET: DO concentration


,target,features,n_features,target_nodes,usable_nodes,all_target_nodes_covered,total_node_hours,minimum_hours_per_node,missing_nodes
0,DO concentration,Water temperature + Specific conductance,2,17,17,True,114607,1885,[]
1,DO concentration,Water temperature,1,17,17,True,115353,1885,[]
2,DO concentration,Specific conductance,1,17,17,True,114608,1885,[]
3,DO concentration,Water temperature + Specific conductance + pH,3,17,16,False,109620,4113,[92116891]
4,DO concentration,Water temperature + pH,2,17,16,False,110231,4140,[92116891]
5,DO concentration,Specific conductance + pH,2,17,16,False,109621,4113,[92116891]
6,DO concentration,pH,1,17,16,False,110232,4140,[92116891]
7,DO concentration,Water temperature + Specific conductance + Tur...,4,17,14,False,80592,1140,"[92116891, 186559442, 203177183]"
8,DO concentration,Water temperature + Specific conductance + Tur...,3,17,14,False,82418,1140,"[92116891, 186559442, 203177183]"
9,DO concentration,Water temperature + Turbidity + pH,3,17,14,False,80962,1140,"[92116891, 186559442, 203177183]"



TARGET: Water temperature


,target,features,n_features,target_nodes,usable_nodes,all_target_nodes_covered,total_node_hours,minimum_hours_per_node,missing_nodes
0,Water temperature,Specific conductance,1,25,24,False,144980,168,[129078768]
1,Water temperature,Specific conductance + pH,2,25,18,False,120839,168,"[47787846, 58738608, 75465016, 92116891, 96751..."
2,Water temperature,pH,1,25,18,False,121821,168,"[47787846, 58738608, 75465016, 92116891, 96751..."
3,Water temperature,DO concentration + Specific conductance,2,25,17,False,114607,1885,"[47787846, 58738608, 75465016, 96751415, 11091..."
4,Water temperature,DO concentration,1,25,17,False,115353,1885,"[47787846, 58738608, 75465016, 96751415, 11091..."
5,Water temperature,DO concentration + Specific conductance + pH,3,25,16,False,109620,4113,"[47787846, 58738608, 75465016, 92116891, 96751..."
6,Water temperature,Specific conductance + Turbidity + pH,3,25,16,False,87966,134,"[47787846, 58738608, 75465016, 92116891, 96751..."
7,Water temperature,DO concentration + pH,2,25,16,False,110231,4140,"[47787846, 58738608, 75465016, 92116891, 96751..."
8,Water temperature,Specific conductance + Turbidity,2,25,16,False,89948,134,"[47787846, 58738608, 75465016, 92116891, 96751..."
9,Water temperature,Turbidity + pH,2,25,16,False,88666,134,"[47787846, 58738608, 75465016, 92116891, 96751..."



TARGET: Specific conductance


,target,features,n_features,target_nodes,usable_nodes,all_target_nodes_covered,total_node_hours,minimum_hours_per_node,missing_nodes
0,Specific conductance,Water temperature,1,25,24,False,144980,168,[129078768]
1,Specific conductance,pH,1,25,19,False,121011,168,"[47787846, 58738608, 75465016, 92116891, 96751..."
2,Specific conductance,Water temperature + pH,2,25,18,False,120839,168,"[47787846, 58738608, 75465016, 92116891, 96751..."
3,Specific conductance,DO concentration + Water temperature,2,25,17,False,114607,1885,"[47787846, 58738608, 75465016, 96751415, 11091..."
4,Specific conductance,Turbidity + pH,2,25,17,False,88095,126,"[47787846, 58738608, 75465016, 92116891, 96751..."
5,Specific conductance,DO concentration,1,25,17,False,114608,1885,"[47787846, 58738608, 75465016, 96751415, 11091..."
6,Specific conductance,Turbidity,1,25,17,False,90077,126,"[47787846, 58738608, 75465016, 92116891, 96751..."
7,Specific conductance,DO concentration + Water temperature + pH,3,25,16,False,109620,4113,"[47787846, 58738608, 75465016, 92116891, 96751..."
8,Specific conductance,Water temperature + Turbidity + pH,3,25,16,False,87966,134,"[47787846, 58738608, 75465016, 92116891, 96751..."
9,Specific conductance,DO concentration + pH,2,25,16,False,109621,4113,"[47787846, 58738608, 75465016, 92116891, 96751..."



TARGET: Turbidity


,target,features,n_features,target_nodes,usable_nodes,all_target_nodes_covered,total_node_hours,minimum_hours_per_node,missing_nodes
0,Turbidity,Specific conductance + pH,2,17,17,True,88095,126,[]
1,Turbidity,Specific conductance,1,17,17,True,90077,126,[]
2,Turbidity,pH,1,17,17,True,88833,126,[]
3,Turbidity,Water temperature + Specific conductance + pH,3,17,16,False,87966,134,[129078768]
4,Turbidity,Water temperature + Specific conductance,2,17,16,False,89948,134,[129078768]
5,Turbidity,Water temperature + pH,2,17,16,False,88666,134,[129078768]
6,Turbidity,Water temperature,1,17,16,False,90672,134,[129078768]
7,Turbidity,DO concentration + Water temperature + Specifi...,4,17,14,False,80592,1140,"[129078768, 133114453, 134075067]"
8,Turbidity,DO concentration + Water temperature + Specifi...,3,17,14,False,82418,1140,"[129078768, 133114453, 134075067]"
9,Turbidity,DO concentration + Water temperature + pH,3,17,14,False,80962,1140,"[129078768, 133114453, 134075067]"



TARGET: pH


,target,features,n_features,target_nodes,usable_nodes,all_target_nodes_covered,total_node_hours,minimum_hours_per_node,missing_nodes
0,pH,Specific conductance,1,19,19,True,121011,168,[]
1,pH,Water temperature + Specific conductance,2,19,18,False,120839,168,[129078768]
2,pH,Water temperature,1,19,18,False,121821,168,[129078768]
3,pH,Specific conductance + Turbidity,2,19,17,False,88095,126,"[186559442, 203177183]"
4,pH,Turbidity,1,19,17,False,88833,126,"[186559442, 203177183]"
5,pH,DO concentration + Water temperature + Specifi...,3,19,16,False,109620,4113,"[129078768, 133114453, 134075067]"
6,pH,Water temperature + Specific conductance + Tur...,3,19,16,False,87966,134,"[129078768, 186559442, 203177183]"
7,pH,DO concentration + Water temperature,2,19,16,False,110231,4140,"[129078768, 133114453, 134075067]"
8,pH,DO concentration + Specific conductance,2,19,16,False,109621,4113,"[129078768, 133114453, 134075067]"
9,pH,Water temperature + Turbidity,2,19,16,False,88666,134,"[129078768, 186559442, 203177183]"


In [6]:
best_feature_rows = []


for target in WQ_VARIABLES:

    target_table = (
        coverage_table[
            coverage_table[
                "target"
            ] == target
        ]
        .copy()
    )


    full_coverage = (
        target_table[
            target_table[
                "all_target_nodes_covered"
            ]
        ]
        .sort_values(
            [
                "n_features",
                "total_node_hours",
                "minimum_hours_per_node"
            ],
            ascending=[
                False,
                False,
                False
            ]
        )
    )


    if len(
        full_coverage
    ) > 0:

        best = (
            full_coverage
            .iloc[0]
        )

        status = (
            "ALL target nodes"
        )

    else:

        # No predictor set reaches every target node.
        # Show maximum possible coverage instead.
        best = (
            target_table
            .sort_values(
                [
                    "usable_nodes",
                    "n_features",
                    "total_node_hours"
                ],
                ascending=[
                    False,
                    False,
                    False
                ]
            )
            .iloc[0]
        )

        status = (
            "Maximum available coverage"
        )


    best_feature_rows.append({

        "target":
            target,

        "target_nodes":
            best[
                "target_nodes"
            ],

        "usable_nodes":
            best[
                "usable_nodes"
            ],

        "predictors":
            best[
                "features"
            ],

        "n_predictors":
            best[
                "n_features"
            ],

        "node_hours":
            best[
                "total_node_hours"
            ],

        "minimum_hours_per_node":
            best[
                "minimum_hours_per_node"
            ],

        "coverage_status":
            status
    })


best_feature_summary = (
    pd.DataFrame(
        best_feature_rows
    )
)


display(
    best_feature_summary
)

,target,target_nodes,usable_nodes,predictors,n_predictors,node_hours,minimum_hours_per_node,coverage_status
0,DO concentration,17,17,Water temperature + Specific conductance,2,114607,1885,ALL target nodes
1,Water temperature,25,24,Specific conductance,1,144980,168,Maximum available coverage
2,Specific conductance,25,24,Water temperature,1,144980,168,Maximum available coverage
3,Turbidity,17,17,Specific conductance + pH,2,88095,126,ALL target nodes
4,pH,19,19,Specific conductance,1,121011,168,ALL target nodes


In [7]:
import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse.linalg import splu


boundary_nodes = sorted(
    all_wq_nodes
)

boundary_index = {
    node: i
    for i, node in enumerate(boundary_nodes)
}

N_BOUNDARY = len(
    boundary_nodes
)


print(
    "Boundary WQ nodes:",
    N_BOUNDARY
)


boundary_full_idx = np.array(
    [
        full_node_index[node]
        for node in boundary_nodes
    ],
    dtype=int
)


all_full_idx = np.arange(
    len(full_node_order)
)


interior_mask = np.ones(
    len(full_node_order),
    dtype=bool
)

interior_mask[
    boundary_full_idx
] = False


interior_full_idx = all_full_idx[
    interior_mask
]


print(
    "Interior/unlabeled nodes:",
    len(interior_full_idx)
)


L_BB = (
    L_full[
        boundary_full_idx,
        :
    ][
        :,
        boundary_full_idx
    ]
    .tocsc()
)


L_BI = (
    L_full[
        boundary_full_idx,
        :
    ][
        :,
        interior_full_idx
    ]
    .tocsc()
)


L_IB = (
    L_full[
        interior_full_idx,
        :
    ][
        :,
        boundary_full_idx
    ]
    .tocsc()
)


L_II = (
    L_full[
        interior_full_idx,
        :
    ][
        :,
        interior_full_idx
    ]
    .tocsc()
)


print(
    "L_BB:",
    L_BB.shape
)

print(
    "L_II:",
    L_II.shape
)


print(
    "\nFactorizing interior Laplacian..."
)

lu = splu(
    L_II
)

print(
    "Factorization complete."
)


rhs = (
    L_IB
    .toarray()
)


solution = lu.solve(
    rhs
)


L_kron = (
    L_BB.toarray()
    -
    L_BI @ solution
)


L_kron = np.asarray(
    L_kron
)


L_kron = (
    L_kron
    +
    L_kron.T
) / 2.0


for i in range(
    N_BOUNDARY
):

    off_diag_sum = (
        L_kron[i, :].sum()
        -
        L_kron[i, i]
    )

    L_kron[
        i,
        i
    ] = -off_diag_sum


positive_diag = np.diag(
    L_kron
)

positive_diag = positive_diag[
    positive_diag > 1e-12
]


LAPLACIAN_SCALE = (
    positive_diag.mean()
)


L_kron_scaled = (
    L_kron
    /
    LAPLACIAN_SCALE
)


print(
    "\nReduced Laplacian shape:",
    L_kron_scaled.shape
)

print(
    "Scale factor:",
    LAPLACIAN_SCALE
)

print(
    "Symmetry error:",
    np.max(
        np.abs(
            L_kron_scaled
            -
            L_kron_scaled.T
        )
    )
)

print(
    "Maximum row-sum error:",
    np.max(
        np.abs(
            L_kron_scaled.sum(
                axis=1
            )
        )
    )
)


eigenvalues = np.linalg.eigvalsh(
    L_kron_scaled
)


print(
    "Smallest eigenvalues:"
)

print(
    eigenvalues[:5]
)


print(
    "\nKron reduction complete."
)

print(
    "The model will use the full 67k-node network"
)

print(
    "through this exact 26-node effective Laplacian."
)

Boundary WQ nodes: 26
Interior/unlabeled nodes: 67375
L_BB: (26, 26)
L_II: (67375, 67375)

Factorizing interior Laplacian...
Factorization complete.

Reduced Laplacian shape: (26, 26)
Scale factor: 0.047553741042819814
Symmetry error: 0.0
Maximum row-sum error: 2.220446049250313e-16
Smallest eigenvalues:
[-2.75278931e-16  5.43830838e-03  8.04522277e-03  1.10802187e-02
  4.11998132e-02]

Kron reduction complete.
The model will use the full 67k-node network
through this exact 26-node effective Laplacian.


In [8]:
TARGET_FEATURES = {

    "DO concentration": [
        "Water temperature",
        "Specific conductance"
    ],

    "Water temperature": [
        "Specific conductance"
    ],

    "Specific conductance": [
        "Water temperature"
    ],

    "Turbidity": [
        "Specific conductance",
        "pH"
    ],

    "pH": [
        "Specific conductance"
    ]
}


target_datasets = {}


for target, features in TARGET_FEATURES.items():

    needed = (
        [
            "node",
            "time",
            target
        ]
        +
        features
    )


    target_df = (
        node_hourly_all[
            needed
        ]
        .dropna(
            subset=[
                target
            ] + features
        )
        .copy()
    )


    target_datasets[
        target
    ] = target_df


    print(
        "\n",
        target
    )

    print(
        "  predictors:",
        features
    )

    print(
        "  nodes:",
        target_df[
            "node"
        ].nunique()
    )

    print(
        "  node-hours:",
        len(
            target_df
        )
    )


 DO concentration
  predictors: ['Water temperature', 'Specific conductance']
  nodes: 17
  node-hours: 114607

 Water temperature
  predictors: ['Specific conductance']
  nodes: 24
  node-hours: 144980

 Specific conductance
  predictors: ['Water temperature']
  nodes: 24
  node-hours: 144980

 Turbidity
  predictors: ['Specific conductance', 'pH']
  nodes: 17
  node-hours: 88095

 pH
  predictors: ['Specific conductance']
  nodes: 19
  node-hours: 121011


In [12]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)


LINEAR_ALPHA_GRID = [
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0
]


GRAPH_LAMBDA_GRID = [
    0.01,
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0,
    30.0,
    100.0
]


GRAPH_BETA_GRID = [
    0.03,
    0.1,
    0.3,
    1.0,
    3.0,
    10.0
]


def make_node_stats(
    df,
    target,
    features
):

    stats = {}


    for node, group in df.groupby(
        "node"
    ):

        X = (
            group[
                features
            ]
            .to_numpy(
                dtype=float
            )
        )

        y = (
            group[
                target
            ]
            .to_numpy(
                dtype=float
            )
        )


        n = len(
            group
        )


        stats[
            int(node)
        ] = {

            "n":
                n,

            "mean_x":
                X.mean(
                    axis=0
                ),

            "exx":
                (
                    X.T @ X
                ) / n,

            "exy":
                (
                    X.T @ y
                ) / n,

            "mean_y":
                y.mean(),

            "mean_y2":
                np.mean(
                    y ** 2
                )
        }


    return stats


def training_scaling(
    stats,
    train_nodes
):

    mean_x = np.mean(
        [
            stats[node][
                "mean_x"
            ]
            for node
            in train_nodes
        ],
        axis=0
    )


    ex2 = np.mean(
        [
            np.diag(
                stats[node][
                    "exx"
                ]
            )
            for node
            in train_nodes
        ],
        axis=0
    )


    variance = (
        ex2
        -
        mean_x ** 2
    )


    variance = np.maximum(
        variance,
        1e-12
    )


    std_x = np.sqrt(
        variance
    )


    return (
        mean_x,
        std_x
    )


def standard_moments(
    stat,
    mean_x,
    std_x
):

    mx = stat[
        "mean_x"
    ]

    exx = stat[
        "exx"
    ]

    exy = stat[
        "exy"
    ]

    my = stat[
        "mean_y"
    ]


    zmean = (
        mx - mean_x
    ) / std_x


    zexx = (

        exx

        - np.outer(
            mx,
            mean_x
        )

        - np.outer(
            mean_x,
            mx
        )

        + np.outer(
            mean_x,
            mean_x
        )

    ) / np.outer(
        std_x,
        std_x
    )


    zexy = (
        exy
        -
        mean_x * my
    ) / std_x


    qmean = np.concatenate(
        (
            [1.0],
            zmean
        )
    )


    p = len(
        mean_x
    )


    q2 = np.zeros(
        (
            p + 1,
            p + 1
        )
    )


    q2[
        0,
        0
    ] = 1.0


    q2[
        0,
        1:
    ] = zmean


    q2[
        1:,
        0
    ] = zmean


    q2[
        1:,
        1:
    ] = zexx


    qy = np.concatenate(
        (
            [my],
            zexy
        )
    )


    return (
        qmean,
        q2,
        qy
    )


def fit_linear_model(
    stats,
    train_nodes,
    alpha
):

    mean_x, std_x = (
        training_scaling(
            stats,
            train_nodes
        )
    )


    p = len(
        mean_x
    )


    A = np.zeros(
        (
            p + 1,
            p + 1
        )
    )


    b = np.zeros(
        p + 1
    )


    for node in train_nodes:

        _, q2, qy = (
            standard_moments(
                stats[node],
                mean_x,
                std_x
            )
        )


        A += q2
        b += qy


    penalty = np.eye(
        p + 1
    )

    penalty[
        0,
        0
    ] = 0.0


    beta = np.linalg.solve(

        A
        +
        alpha
        * penalty,

        b
    )


    return {

        "beta":
            beta,

        "mean_x":
            mean_x,

        "std_x":
            std_x
    }


def fit_graph_model(
    stats,
    train_nodes,
    lambda_lap,
    lambda_beta
):

    mean_x, std_x = (
        training_scaling(
            stats,
            train_nodes
        )
    )


    p = len(
        mean_x
    )


    A11 = np.zeros(
        (
            p + 1,
            p + 1
        )
    )


    A12 = np.zeros(
        (
            p + 1,
            N_BOUNDARY
        )
    )


    rhs_beta = np.zeros(
        p + 1
    )


    mask = np.zeros(
        N_BOUNDARY
    )


    rhs_g = np.zeros(
        N_BOUNDARY
    )


    for node in train_nodes:

        qmean, q2, qy = (
            standard_moments(
                stats[node],
                mean_x,
                std_x
            )
        )


        j = boundary_index[
            node
        ]


        A11 += q2

        A12[
            :,
            j
        ] += qmean

        rhs_beta += qy

        mask[
            j
        ] = 1.0

        rhs_g[
            j
        ] = stats[
            node
        ][
            "mean_y"
        ]


    ridge = np.eye(
        p + 1
    )

    ridge[
        0,
        0
    ] = 0.0


    A11 += (
        lambda_beta
        * ridge
    )


    A22 = (
        np.diag(
            mask
        )

        +
        lambda_lap
        * L_kron_scaled
    )


    size = (
        p + 1
        +
        N_BOUNDARY
        +
        1
    )


    A = np.zeros(
        (
            size,
            size
        )
    )


    rhs = np.zeros(
        size
    )


    q = p + 1


    A[
        :q,
        :q
    ] = A11


    A[
        :q,
        q:q + N_BOUNDARY
    ] = A12


    A[
        q:q + N_BOUNDARY,
        :q
    ] = A12.T


    A[
        q:q + N_BOUNDARY,
        q:q + N_BOUNDARY
    ] = A22


    A[
        q:q + N_BOUNDARY,
        -1
    ] = 1.0


    A[
        -1,
        q:q + N_BOUNDARY
    ] = 1.0


    rhs[
        :q
    ] = rhs_beta


    rhs[
        q:q + N_BOUNDARY
    ] = rhs_g


    solution = np.linalg.solve(
        A,
        rhs
    )


    beta = solution[
        :q
    ]


    g = solution[
        q:q + N_BOUNDARY
    ]


    return {

        "beta":
            beta,

        "g":
            g,

        "mean_x":
            mean_x,

        "std_x":
            std_x
    }


def node_mse(
    node,
    stat,
    model,
    use_graph=False
):

    qmean, q2, qy = (
        standard_moments(

            stat,

            model[
                "mean_x"
            ],

            model[
                "std_x"
            ]
        )
    )


    coef = model[
        "beta"
    ].copy()


    if use_graph:

        coef[
            0
        ] += (
            model["g"][
                boundary_index[
                    node
                ]
            ]
        )


    mse = (

        stat[
            "mean_y2"
        ]

        - 2.0
        * coef.dot(
            qy
        )

        + coef.dot(
            q2 @ coef
        )
    )


    return max(
        float(mse),
        0.0
    )


def tune_inner_loo(
    stats,
    outer_train_nodes
):

    linear_scores = {}


    for alpha in LINEAR_ALPHA_GRID:

        fold_mse = []


        for validation_node in outer_train_nodes:

            train_nodes = [
                node
                for node
                in outer_train_nodes
                if node != validation_node
            ]


            model = fit_linear_model(
                stats,
                train_nodes,
                alpha
            )


            fold_mse.append(

                node_mse(
                    validation_node,
                    stats[
                        validation_node
                    ],
                    model,
                    use_graph=False
                )

            )


        linear_scores[
            alpha
        ] = np.mean(
            fold_mse
        )


    best_alpha = min(
        linear_scores,
        key=
            linear_scores.get
    )


    graph_scores = {}


    for lambda_lap in GRAPH_LAMBDA_GRID:

        for lambda_beta in GRAPH_BETA_GRID:

            fold_mse = []


            for validation_node in outer_train_nodes:

                train_nodes = [

                    node

                    for node
                    in outer_train_nodes

                    if node != validation_node

                ]


                model = fit_graph_model(

                    stats,

                    train_nodes,

                    lambda_lap=
                        lambda_lap,

                    lambda_beta=
                        lambda_beta
                )


                fold_mse.append(

                    node_mse(

                        validation_node,

                        stats[
                            validation_node
                        ],

                        model,

                        use_graph=True
                    )

                )


            graph_scores[
                (
                    lambda_lap,
                    lambda_beta
                )
            ] = np.mean(
                fold_mse
            )


    best_graph = min(
        graph_scores,
        key=
            graph_scores.get
    )


    return (
        best_alpha,
        best_graph
    )


def predict_rows(
    df,
    features,
    model,
    graph=False
):

    X = (
        df[
            features
        ]
        .to_numpy(
            dtype=float
        )
    )


    Z = (

        X
        -
        model[
            "mean_x"
        ]

    ) / model[
        "std_x"
    ]


    Q = np.column_stack(
        (
            np.ones(
                len(df)
            ),
            Z
        )
    )


    pred = (
        Q
        @ model[
            "beta"
        ]
    )


    if graph:

        effects = np.array(
            [
                model["g"][
                    boundary_index[
                        int(node)
                    ]
                ]

                for node
                in df["node"]
            ]
        )


        pred += effects


    return pred


def nested_loo_full_graph(
    df,
    target,
    features
):

    stats = make_node_stats(
        df,
        target,
        features
    )


    usable_nodes = sorted(
        stats.keys()
    )


    print(
        "\n",
        "=" * 70
    )

    print(
        "TARGET:",
        target
    )

    print(
        "Predictors:",
        features
    )

    print(
        "LOO nodes:",
        len(
            usable_nodes
        )
    )

    print(
        "Full graph nodes:",
        len(
            full_node_order
        )
    )


    prediction_parts = []

    tuning_rows = []


    for fold, test_node in enumerate(
        usable_nodes,
        start=1
    ):

        print(
            f"  Outer fold {fold}/{len(usable_nodes)} "
            f"- node {test_node}"
        )


        train_nodes = [

            node

            for node
            in usable_nodes

            if node != test_node

        ]


        (
            best_linear_alpha,
            best_graph
        ) = tune_inner_loo(

            stats,
            train_nodes
        )


        (
            best_lambda_lap,
            best_lambda_beta
        ) = best_graph


        linear_model = fit_linear_model(

            stats,

            train_nodes,

            best_linear_alpha
        )


        graph_model = fit_graph_model(

            stats,

            train_nodes,

            lambda_lap=
                best_lambda_lap,

            lambda_beta=
                best_lambda_beta
        )


        test_df = (
            df[
                df["node"]
                == test_node
            ]
            .copy()
        )


        linear_pred = predict_rows(

            test_df,

            features,

            linear_model,

            graph=False
        )


        graph_pred = predict_rows(

            test_df,

            features,

            graph_model,

            graph=True
        )


        fold_df = pd.DataFrame({

            "node":
                test_df[
                    "node"
                ].to_numpy(),

            "target":
                target,

            "y_true":
                test_df[
                    target
                ].to_numpy(),

            "linear_prediction":
                linear_pred,

            "graph_prediction":
                graph_pred
        })


        prediction_parts.append(
            fold_df
        )


        tuning_rows.append({

            "heldout_node":
                test_node,

            "linear_alpha":
                best_linear_alpha,

            "graph_lambda_lap":
                best_lambda_lap,

            "graph_lambda_beta":
                best_lambda_beta
        })


    predictions = pd.concat(
        prediction_parts,
        ignore_index=True
    )


    counts = (
        predictions
        .groupby(
            "node"
        )
        .size()
    )


    predictions[
        "weight"
    ] = (
        predictions[
            "node"
        ]
        .map(
            1.0
            / counts
        )
    )


    w = predictions[
        "weight"
    ].to_numpy()


    y = predictions[
        "y_true"
    ].to_numpy()


    lp = predictions[
        "linear_prediction"
    ].to_numpy()


    gp = predictions[
        "graph_prediction"
    ].to_numpy()


    linear_rmse = np.sqrt(
        mean_squared_error(
            y,
            lp,
            sample_weight=w
        )
    )


    graph_rmse = np.sqrt(
        mean_squared_error(
            y,
            gp,
            sample_weight=w
        )
    )


    linear_r2 = r2_score(
        y,
        lp,
        sample_weight=w
    )


    graph_r2 = r2_score(
        y,
        gp,
        sample_weight=w
    )


    result = {

        "Target":
            target,

        "Nodes":
            len(
                usable_nodes
            ),

        "Linear R2":
            linear_r2,

        "Graph R2":
            graph_r2,

        "Linear RMSE":
            linear_rmse,

        "Graph RMSE":
            graph_rmse
    }


    return (
        result,
        predictions,
        pd.DataFrame(
            tuning_rows
        )
    )

In [13]:
full_graph_results = []

full_graph_predictions = {}

full_graph_tuning = {}


for target, features in TARGET_FEATURES.items():

    df = target_datasets[
        target
    ]


    result, predictions, tuning = (
        nested_loo_full_graph(

            df=df,

            target=target,

            features=features
        )
    )


    full_graph_results.append(
        result
    )


    full_graph_predictions[
        target
    ] = predictions


    full_graph_tuning[
        target
    ] = tuning


full_graph_results = pd.DataFrame(
    full_graph_results
)


target_order = [
    "DO concentration",
    "Water temperature",
    "pH",
    "Specific conductance",
    "Turbidity"
]


full_graph_results[
    "Target"
] = pd.Categorical(

    full_graph_results[
        "Target"
    ],

    categories=
        target_order,

    ordered=True
)


full_graph_results = (

    full_graph_results

    .sort_values(
        "Target"
    )

    .reset_index(
        drop=True
    )
)


display(
    full_graph_results[
        [
            "Target",
            "Nodes",
            "Linear R2",
            "Graph R2",
            "Linear RMSE",
            "Graph RMSE"
        ]
    ]
    .style
    .format({

        "Linear R2":
            "{:.3f}",

        "Graph R2":
            "{:.3f}",

        "Linear RMSE":
            "{:.4f}",

        "Graph RMSE":
            "{:.4f}"
    })
)


full_graph_results.to_csv(
    "full_graph_nested_loo_results.csv",
    index=False
)


TARGET: DO concentration
Predictors: ['Water temperature', 'Specific conductance']
LOO nodes: 17
Full graph nodes: 67401
  Outer fold 1/17 - node 67739434
  Outer fold 2/17 - node 81497938
  Outer fold 3/17 - node 82982643
  Outer fold 4/17 - node 83027786
  Outer fold 5/17 - node 83222751
  Outer fold 6/17 - node 84197086
  Outer fold 7/17 - node 88786999
  Outer fold 8/17 - node 90706215
  Outer fold 9/17 - node 92116891
  Outer fold 10/17 - node 93300040
  Outer fold 11/17 - node 93392084
  Outer fold 12/17 - node 96496279
  Outer fold 13/17 - node 97532807
  Outer fold 14/17 - node 126377584
  Outer fold 15/17 - node 131520862
  Outer fold 16/17 - node 186559442
  Outer fold 17/17 - node 203177183

TARGET: Water temperature
Predictors: ['Specific conductance']
LOO nodes: 24
Full graph nodes: 67401
  Outer fold 1/24 - node 47787846
  Outer fold 2/24 - node 58738608
  Outer fold 3/24 - node 67739434
  Outer fold 4/24 - node 75465016
  Outer fold 5/24 - node 81497938
  Outer fold 6/2

,Target,Nodes,Linear R2,Graph R2,Linear RMSE,Graph RMSE
0,DO concentration,17,0.621,0.613,1.0921,1.1026
1,Water temperature,24,-0.056,-0.035,7.2828,7.2100
2,pH,19,-0.035,0.074,0.2411,0.2280
3,Specific conductance,24,-0.089,0.091,462.9602,423.0154
4,Turbidity,17,0.023,-0.049,559.4347,579.5058


In [14]:
from scipy import sparse
from scipy.sparse.linalg import splu


compressed_node_order = list(
    G_master.nodes()
)

compressed_node_index = {
    node: i
    for i, node
    in enumerate(compressed_node_order)
}

N_COMPRESSED = len(
    compressed_node_order
)


rows = []
cols = []
weights = []


G_master_und = G_master.to_undirected(
    as_view=True
)


for u, v, data in G_master_und.edges(
    data=True
):

    length_m = data.get(
        "length_m",
        None
    )


    if (
        length_m is None
        or not np.isfinite(float(length_m))
        or float(length_m) <= 0
    ):

        udata = G_master.nodes[u]
        vdata = G_master.nodes[v]

        length_m = haversine_m(
            udata["longitude"],
            udata["latitude"],
            vdata["longitude"],
            vdata["latitude"]
        )


    distance_km = max(
        float(length_m) / 1000.0,
        1e-8
    )


    w = (
        1.0
        /
        distance_km
    )


    i = compressed_node_index[u]
    j = compressed_node_index[v]


    rows.extend([
        i,
        j
    ])

    cols.extend([
        j,
        i
    ])

    weights.extend([
        w,
        w
    ])


W_compressed = sparse.coo_matrix(
    (
        weights,
        (
            rows,
            cols
        )
    ),
    shape=(
        N_COMPRESSED,
        N_COMPRESSED
    )
).tocsr()


compressed_degree = np.asarray(
    W_compressed.sum(axis=1)
).ravel()


L_compressed = (
    sparse.diags(
        compressed_degree
    )
    -
    W_compressed
).tocsr()


print(
    "Compressed Laplacian:",
    L_compressed.shape
)

print(
    "Nonzero entries:",
    L_compressed.nnz
)

Compressed Laplacian: (67, 67)
Nonzero entries: 199


In [15]:
compressed_boundary_nodes = sorted(
    all_wq_nodes
)


missing_from_compressed = (
    set(compressed_boundary_nodes)
    -
    set(compressed_node_order)
)


print(
    "Missing WQ nodes from compressed graph:",
    missing_from_compressed
)


compressed_boundary_idx = np.array(
    [
        compressed_node_index[node]
        for node
        in compressed_boundary_nodes
    ],
    dtype=int
)


all_compressed_idx = np.arange(
    N_COMPRESSED
)


compressed_interior_mask = np.ones(
    N_COMPRESSED,
    dtype=bool
)

compressed_interior_mask[
    compressed_boundary_idx
] = False


compressed_interior_idx = all_compressed_idx[
    compressed_interior_mask
]


C_BB = (
    L_compressed[
        compressed_boundary_idx,
        :
    ][
        :,
        compressed_boundary_idx
    ]
    .tocsc()
)


C_BI = (
    L_compressed[
        compressed_boundary_idx,
        :
    ][
        :,
        compressed_interior_idx
    ]
    .tocsc()
)


C_IB = (
    L_compressed[
        compressed_interior_idx,
        :
    ][
        :,
        compressed_boundary_idx
    ]
    .tocsc()
)


C_II = (
    L_compressed[
        compressed_interior_idx,
        :
    ][
        :,
        compressed_interior_idx
    ]
    .tocsc()
)


compressed_lu = splu(
    C_II
)


compressed_solution = compressed_lu.solve(
    C_IB.toarray()
)


L_kron_compressed = (
    C_BB.toarray()
    -
    C_BI @ compressed_solution
)


L_kron_compressed = np.asarray(
    L_kron_compressed
)


L_kron_compressed = (
    L_kron_compressed
    +
    L_kron_compressed.T
) / 2.0


for i in range(
    len(compressed_boundary_nodes)
):

    off_diag_sum = (
        L_kron_compressed[i, :].sum()
        -
        L_kron_compressed[i, i]
    )

    L_kron_compressed[
        i,
        i
    ] = -off_diag_sum


compressed_positive_diag = np.diag(
    L_kron_compressed
)

compressed_positive_diag = compressed_positive_diag[
    compressed_positive_diag > 1e-12
]


COMPRESSED_SCALE = (
    compressed_positive_diag.mean()
)


L_kron_compressed_scaled = (
    L_kron_compressed
    /
    COMPRESSED_SCALE
)


print(
    "Compressed effective Laplacian:",
    L_kron_compressed_scaled.shape
)

print(
    "Maximum row-sum error:",
    np.max(
        np.abs(
            L_kron_compressed_scaled.sum(
                axis=1
            )
        )
    )
)

Missing WQ nodes from compressed graph: set()
Compressed effective Laplacian: (26, 26)
Maximum row-sum error: 4.440892098500626e-16


In [16]:
difference = (
    L_kron_scaled
    -
    L_kron_compressed_scaled
)


absolute_max_difference = np.max(
    np.abs(
        difference
    )
)


frobenius_difference = np.linalg.norm(
    difference,
    ord="fro"
)


relative_frobenius_difference = (
    frobenius_difference
    /
    np.linalg.norm(
        L_kron_scaled,
        ord="fro"
    )
)


print(
    "Maximum absolute matrix difference:",
    absolute_max_difference
)

print(
    "Frobenius difference:",
    frobenius_difference
)

print(
    "Relative Frobenius difference:",
    relative_frobenius_difference
)

Maximum absolute matrix difference: 2.807531984672096e-12
Frobenius difference: 6.588950514700009e-12
Relative Frobenius difference: 5.656678447544951e-13


In [17]:
def fit_graph_model_with_L(
    stats,
    train_nodes,
    lambda_lap,
    lambda_beta,
    effective_L
):

    mean_x, std_x = (
        training_scaling(
            stats,
            train_nodes
        )
    )


    p = len(
        mean_x
    )


    A11 = np.zeros(
        (
            p + 1,
            p + 1
        )
    )


    A12 = np.zeros(
        (
            p + 1,
            N_BOUNDARY
        )
    )


    rhs_beta = np.zeros(
        p + 1
    )


    mask = np.zeros(
        N_BOUNDARY
    )


    rhs_g = np.zeros(
        N_BOUNDARY
    )


    for node in train_nodes:

        qmean, q2, qy = (
            standard_moments(
                stats[node],
                mean_x,
                std_x
            )
        )


        j = boundary_index[
            node
        ]


        A11 += q2

        A12[:, j] += qmean

        rhs_beta += qy

        mask[j] = 1.0

        rhs_g[j] = (
            stats[node][
                "mean_y"
            ]
        )


    ridge = np.eye(
        p + 1
    )

    ridge[
        0,
        0
    ] = 0.0


    A11 += (
        lambda_beta
        *
        ridge
    )


    A22 = (
        np.diag(
            mask
        )
        +
        lambda_lap
        *
        effective_L
    )


    q = p + 1


    size = (
        q
        +
        N_BOUNDARY
        +
        1
    )


    A = np.zeros(
        (
            size,
            size
        )
    )


    rhs = np.zeros(
        size
    )


    A[
        :q,
        :q
    ] = A11


    A[
        :q,
        q:q + N_BOUNDARY
    ] = A12


    A[
        q:q + N_BOUNDARY,
        :q
    ] = A12.T


    A[
        q:q + N_BOUNDARY,
        q:q + N_BOUNDARY
    ] = A22


    A[
        q:q + N_BOUNDARY,
        -1
    ] = 1.0


    A[
        -1,
        q:q + N_BOUNDARY
    ] = 1.0


    rhs[
        :q
    ] = rhs_beta


    rhs[
        q:q + N_BOUNDARY
    ] = rhs_g


    solution = np.linalg.solve(
        A,
        rhs
    )


    return {
        "beta":
            solution[:q],

        "g":
            solution[
                q:q + N_BOUNDARY
            ],

        "mean_x":
            mean_x,

        "std_x":
            std_x
    }

In [18]:
def tune_inner_loo_with_L(
    stats,
    outer_train_nodes,
    effective_L
):

    linear_scores = {}


    for alpha in LINEAR_ALPHA_GRID:

        errors = []


        for validation_node in outer_train_nodes:

            train_nodes = [
                node
                for node
                in outer_train_nodes
                if node != validation_node
            ]


            model = fit_linear_model(
                stats,
                train_nodes,
                alpha
            )


            errors.append(
                node_mse(
                    validation_node,
                    stats[
                        validation_node
                    ],
                    model,
                    use_graph=False
                )
            )


        linear_scores[
            alpha
        ] = np.mean(
            errors
        )


    best_alpha = min(
        linear_scores,
        key=linear_scores.get
    )


    graph_scores = {}


    for lambda_lap in GRAPH_LAMBDA_GRID:

        for lambda_beta in GRAPH_BETA_GRID:

            errors = []


            for validation_node in outer_train_nodes:

                train_nodes = [
                    node
                    for node
                    in outer_train_nodes
                    if node != validation_node
                ]


                model = fit_graph_model_with_L(
                    stats=stats,
                    train_nodes=train_nodes,
                    lambda_lap=lambda_lap,
                    lambda_beta=lambda_beta,
                    effective_L=effective_L
                )


                errors.append(
                    node_mse(
                        validation_node,
                        stats[
                            validation_node
                        ],
                        model,
                        use_graph=True
                    )
                )


            graph_scores[
                (
                    lambda_lap,
                    lambda_beta
                )
            ] = np.mean(
                errors
            )


    best_graph = min(
        graph_scores,
        key=graph_scores.get
    )


    return (
        best_alpha,
        best_graph
    )

# New Edge Weights